# arora_data functions

Each cell calls one function against prod, read-only, as `readonly_user`, and shows its output. Change the arguments and re-run any cell.

In [ ]:
from datetime import date, timedelta

import polars as pl

from IPython.display import Markdown

from arora_data import _readers
from arora_data.dates import today_et
from arora_data.forex import FxBar
from arora_data.frames import to_df
from arora_data.series import Series
from arora_data.sync import _sync, check_fresh, forex, list_columns, list_schemas, list_series, list_tables

# The two readers are private: apps will call named series functions instead. Wrapped here only to explore.
read_prices = _sync(_readers._read_prices)
read_values = _sync(_readers._read_values)


def show(rows):
    """One result or a list of results as a Markdown table."""
    rows = [r.model_dump() for r in (rows if isinstance(rows, list) else [rows])]
    if not rows:
        return Markdown("0 rows")
    head = list(rows[0])
    lines = ["|".join(head), "|".join("---" for _ in head)]
    lines += ["|".join(str(r[c]).replace("|", "/") for c in head) for r in rows]
    return Markdown(f"{len(rows)} rows\n\n" + "\n".join(lines))

## `today_et()`
Today's date in US Eastern time.

In [ ]:
today_et()

## `list_schemas()`
The data schemas and what each holds.

In [ ]:
show(list_schemas())

## `list_tables(schema)`
Each table's description, estimated rows and most recent date.

In [ ]:
show(list_tables("forex"))

## `list_columns(schema, table)`
Column names, types and whether they can be blank.

In [ ]:
show(list_columns("forex", "prices"))

## `list_series(schema, table)`
The series in a time-series table, with each one's most recent date.

In [ ]:
show(list_series("forex", "prices"))

## `check_fresh(series)` on a current series
USD/CAD is daily, so it is stale after 10 days without a new point.

In [ ]:
show(check_fresh(Series("forex.prices", "USDCAD", frequency="daily")))

## `check_fresh(series)` on a stopped series
FRED `GDP` was dropped from the FRED list in migration 096, and has no frequency label, so it counts as stale.

In [ ]:
show(check_fresh(Series("macro.observations", "GDP")))

## `forex(base, quote, start, end)` on a stored pair
FMP daily bars, returned as stored (`route` = stored).

In [ ]:
show(forex("USD", "CAD", today_et() - timedelta(days=7)))

## `forex(...)` on an inverted pair
Only USDCAD is stored, so CAD/USD is 1 / USDCAD. High and low swap. volume is FMP's tick count, the same either way round; vwap is 1 / vwap, a close approximation.

In [ ]:
show(forex("CAD", "USD", today_et() - timedelta(days=7)))

## `forex(...)` on a crossed pair
BRL/INR isn't stored, so it is usd_per(BRL) / usd_per(INR), from USDBRL and USDINR. A cross has open and close only, and skips a date either USD pair lacks.

In [ ]:
show(forex("BRL", "INR", today_et() - timedelta(days=7)))

## `forex("ALL", "ALL", start, end)`
Every pair of stored currencies once: stored pairs as stored, the rest crossed. Keep the range short: one week is about 50,000 rows.

In [ ]:
to_df(forex("ALL", "ALL", today_et() - timedelta(days=3)), FxBar)

# Showcase: `forex()`
A few ways to use it. Results come back as `FxBar` rows; `to_df` turns them into a Polars frame with exact decimals.

## Full history in one call
EUR/USD from 1999, the start of FMP's history. There is no row limit, so the whole series comes back.

In [ ]:
eurusd = to_df(forex("EUR", "USD", date(1999, 1, 1)), FxBar)
print(f"{eurusd.height:,} days, {eurusd['date'].min()} to {eurusd['date'].max()}")
eurusd.select("date", "open", "high", "low", "close").tail(5)

## Year-end closes
The last close of each year, from the frame above.

In [ ]:
(
    eurusd.group_by(pl.col("date").dt.year().alias("year"), maintain_order=True)
    .agg(pl.col("date").last().alias("last_day"), pl.col("close").last())
    .tail(10)
)

## Several currencies in rupees, side by side
What one unit of each costs in INR. USD/INR is stored; the others are crossed through USD.

In [ ]:
start = today_et() - timedelta(days=14)
rows = [r for base in ["USD", "EUR", "GBP", "JPY", "CNY"] for r in forex(base, "INR", start)]
to_df(rows, FxBar).pivot(on="pair", index="date", values="close")

## An exotic cross
One Kuwaiti dinar in Lebanese pounds. Computed rates keep 12 significant digits, so large and tiny crosses both stay exact in a dataframe.

In [ ]:
show(forex("KWD", "LBP", today_et() - timedelta(days=7)))

## A currency with no USD pair
Gold (XAU), silver, IMF units and retired currencies have no stored USD pair, so they raise a clear error instead of returning nothing.

In [ ]:
try:
    forex("XAU", "USD", today_et() - timedelta(days=7))
except ValueError as err:
    print(err)

## `_read_prices(table, symbol, start, end)`
Daily price bars, oldest first. Private: for exploring only.

In [ ]:
show(read_prices("forex.prices", "USDCAD", today_et() - timedelta(days=14), today_et()))

## `_read_values(table, key, start, end)`
One series' values, oldest first. Private: for exploring only.

In [ ]:
show(read_values("macro.observations", "BAMLH0A0HYM2", today_et() - timedelta(days=14), today_et()))

## `_read_values(..., dataset=...)` on an `indicators.*` table
These tables need `dataset`, because the same key can mean different series in different datasets.

In [ ]:
show(read_values("indicators.bls", "CES0000000001", today_et() - timedelta(days=365), today_et(), dataset="bls/core"))

## `to_df(rows, model)`
Any result as a Polars dataframe, with exact `Decimal` values.

In [ ]:
to_df(read_values("macro.observations", "BAMLH0A0HYM2", today_et() - timedelta(days=14), today_et()), _readers.Value)